# BirdCLEF+ 2026 — SED + Perch Ensemble

Blends two independent model families:
- **Perch v2** (0.908 LB): Frozen foundation model + LogReg probes + site/hour priors
- **SED PCEN v13** (0.773 LB): EfficientNet-B0 CNN + attention pooling + temporal smoothing

Species-aware blending:
- 203 Perch-mapped species: `alpha * perch + (1 - alpha) * sed`
- 31 unmapped species (insect sonotypes): higher SED weight since Perch has no signal


In [ ]:
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

# ── Load sample submission (canonical row_id list) ────────────────────────
sample_sub = pd.read_csv(
    sorted(Path('/kaggle/input').rglob('sample_submission.csv'))[0]
)
species_cols = [c for c in sample_sub.columns if c != 'row_id']
N_ROWS = len(sample_sub)
N_CLASSES = len(species_cols)
UNIFORM = 1.0 / N_CLASSES
print(f'Expected: {N_ROWS} rows, {N_CLASSES} species')

# ── Locate submission CSVs from kernel outputs ─────────────────────────────
def find_submission(pattern):
    """Find submission.csv under /kaggle/input matching a path pattern."""
    hits = subprocess.run(
        ['find', '/kaggle/input', '-name', 'submission.csv', '-path', f'*{pattern}*'],
        capture_output=True, text=True
    ).stdout.strip().splitlines()
    hits = [h for h in hits if h]
    assert hits, f'No submission.csv found for pattern: {pattern}'
    print(f'{pattern}: {hits[0]}')
    return hits[0]

def load_and_align(path):
    """Load a submission CSV and align to sample_submission row_ids.
    Missing rows get uniform score."""
    raw = pd.read_csv(path)
    print(f'  Raw shape: {raw.shape}')
    aligned = sample_sub[['row_id']].merge(raw, on='row_id', how='left')
    aligned[species_cols] = aligned[species_cols].fillna(UNIFORM)
    return aligned[species_cols].values.astype(np.float32)

perch_path = find_submission('perch-inference')
sed_path   = find_submission('sed-inference')

P = load_and_align(perch_path)
S = load_and_align(sed_path)

# Check coverage
perch_coverage = (P.max(axis=1) != UNIFORM).sum()
sed_coverage   = (S.max(axis=1) != UNIFORM).sum()
print(f'\nPerch covers {perch_coverage}/{N_ROWS} rows')
print(f'SED covers   {sed_coverage}/{N_ROWS} rows')

In [ ]:
# ── Identify Perch-unmapped species (insect sonotypes) ────────────────────
# Insect sonotypes have labels like '47158son01', '47158son02', etc.
# These have near-zero Perch predictions (logit set to -8.0 in Perch notebook)
unmapped_mask = np.array(['son' in c for c in species_cols], dtype=bool)

# Also detect species where Perch max output is near-zero
perch_max_per_species = P.max(axis=0)
near_zero_mask = perch_max_per_species < 0.001
unmapped_mask = unmapped_mask | near_zero_mask

n_unmapped = unmapped_mask.sum()
n_mapped   = (~unmapped_mask).sum()
print(f'Perch-mapped species:   {n_mapped}')
print(f'Perch-unmapped species: {n_unmapped}')

In [ ]:
# ── Species-aware ensemble ────────────────────────────────────────────────
# Perch (0.908) is much stronger than SED (0.773) on mapped species.
# For unmapped species, SED is the only real signal.
# For rows where SED has no predictions (uniform), use Perch only and vice versa.

ALPHA_MAPPED   = 0.85  # Perch weight for mapped species
ALPHA_UNMAPPED = 0.15  # Perch weight for unmapped species (mostly noise)

alpha = np.where(unmapped_mask, ALPHA_UNMAPPED, ALPHA_MAPPED).astype(np.float32)

# Detect rows where one model has no real predictions (uniform fill)
sed_has_pred   = (S.max(axis=1) > UNIFORM + 0.001)
perch_has_pred = (P.max(axis=1) > UNIFORM + 0.001)

# Blend: alpha * perch + (1 - alpha) * sed
blended = alpha[None, :] * P + (1.0 - alpha[None, :]) * S

# Where only one model has predictions, use that model alone
perch_only = perch_has_pred & ~sed_has_pred
sed_only   = sed_has_pred & ~perch_has_pred
blended[perch_only] = P[perch_only]
blended[sed_only]   = S[sed_only]

print(f'Both models:  {(sed_has_pred & perch_has_pred).sum()} rows')
print(f'Perch only:   {perch_only.sum()} rows')
print(f'SED only:     {sed_only.sum()} rows')
print(f'Neither:      {(~sed_has_pred & ~perch_has_pred).sum()} rows')
print(f'Blended mean: {blended.mean():.4f}')

In [ ]:
# ── Build & save submission ──────────────────────────────────────────────
sub = sample_sub[['row_id']].copy()
sub[species_cols] = blended

assert sub.shape == sample_sub.shape, f'Shape mismatch: {sub.shape} vs {sample_sub.shape}'
assert not sub.isna().any().any(), 'NaNs detected'

out_path = Path('/kaggle/working/submission.csv')
sub.to_csv(out_path, index=False)

print(f'Submission shape: {sub.shape}')
print(f'Saved → {out_path}')
sub.head(3)